# SkyVault — ControlNet SDXL Generation
Runs Stage 05 of the pipeline on a Colab T4 GPU.
Outputs are saved to your Google Drive under `MyDrive/skyvault/`.

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime type to T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update the skyvault repo
import os

REPO_URL = "https://github.com/yakirEng/skyvault.git"
REPO_DIR = "/content/skyvault"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
!pip install -q diffusers>=0.27 transformers accelerate opencv-python pyyaml

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
N_SAMPLES     = 100   # how many layouts to generate and render this session
START_SEED    = 0     # change to resume from a different seed batch
OUTPUT_DIR    = "/content/drive/MyDrive/skyvault/generation_output"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import json
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

from pipeline.config import load_config
from pipeline.layout.generator import generate_layout, export_mask
from pipeline.generation.controlnet_pipeline import (
    load_pipeline, run_generation
)

cfg = load_config("pipeline_config.yaml")
print("Config loaded — canvas:", cfg.resolution.canvas_px, "px, GSD:", cfg.resolution.gsd_cm, "cm/px")

In [ ]:
# Create output directories
out = Path(OUTPUT_DIR)
scenes_dir = out / "scenes"
masks_dir  = out / "masks"
conds_dir  = out / "conditions"

for d in [scenes_dir, masks_dir, conds_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Output directory:", out)

In [ ]:
# Load SDXL + ControlNet pipeline (downloads ~8GB on first run)
print("Loading pipeline... (this takes ~5 min on first run)")
pipeline = load_pipeline(device="cuda")
print("Pipeline ready.")

In [ ]:
# Generation loop
results_log = []

for i in range(N_SAMPLES):
    seed = START_SEED + i
    idx  = f"{seed:04d}"

    # Skip if already generated (allows resuming)
    if (scenes_dir / f"scene_{idx}.png").exists():
        print(f"[{i+1}/{N_SAMPLES}] seed={seed} already exists, skipping.")
        continue

    # Stage 02: layout
    layout = generate_layout(seed=seed, cfg=cfg)

    # Stage 05: ControlNet generation
    result = run_generation(layout, pipeline, cfg)

    # Save scene (RGB PNG)
    Image.fromarray(result.scene).save(scenes_dir / f"scene_{idx}.png")

    # Save normalized mask (single-channel, values 0-4)
    normalized_mask = export_mask(layout.canvas)
    cv2.imwrite(str(masks_dir / f"mask_{idx}.png"), normalized_mask)

    # Save conditioning images for debugging
    Image.fromarray(result.seg_condition).save(conds_dir / f"seg_{idx}.png")
    Image.fromarray(result.edge_condition).save(conds_dir / f"edge_{idx}.png")

    # Save patch metadata (bboxes + class_ids; images saved separately)
    patch_meta = [
        {"class_id": p.class_id, "bbox": list(p.bbox)}
        for p in result.patches
    ]
    with open(out / f"patches_{idx}.json", "w") as f:
        json.dump({
            "seed": seed,
            "classes_present": layout.classes_present,
            "patches": patch_meta,
        }, f, indent=2)

    results_log.append({"seed": seed, "n_patches": len(result.patches)})
    print(f"[{i+1}/{N_SAMPLES}] seed={seed} — {len(result.patches)} patches saved.")

print(f"\nDone. {len(results_log)} new samples saved to {OUTPUT_DIR}")

In [ ]:
# Quick visual check — display first 4 generated scenes
import matplotlib.pyplot as plt

scene_files = sorted(scenes_dir.glob("*.png"))[:4]
fig, axes = plt.subplots(1, len(scene_files), figsize=(20, 5))

for ax, path in zip(axes, scene_files):
    ax.imshow(Image.open(path))
    ax.set_title(path.stem)
    ax.axis("off")

plt.tight_layout()
plt.show()